In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.special import softmax

SEED = 42
rng = np.random.default_rng(SEED)

N_ITEMS = 20
ITEMS_PER_TASK = 4
TASKS_PER_VERSION = 12
N_VERSIONS = 20
N_RESPONDENTS = 1000
ITEM_NAMES = [f"Feature_{i+1:02d}" for i in range(N_ITEMS)]

# Prior utilities
# If we know nothing about preferences: beta = 0
# Later this can be replaced with utilities from:
# - previous MaxDiff study
# - pilot
# - expert prior
# - Bayesian posterior
PRIOR_BETA = np.zeros(N_ITEMS)

# Effect Coding:
# Utility is relative. We estimate J-1 parameters and use the final item as the implied reference/effect-coded item.
def effect_vector(item, n_items):
    x = np.zeros(n_items - 1)
    if item < n_items - 1:
        x[item] = 1
    else:
        x[:] = -1
    return x

# Information matrix for one Maxdiff set
# Sequential Best-Worst model:
#   1. Best selected from K alternatives
#   2. Worst selected from remaining K-1
# We calculate expected Fisher information over all possible best choices.
def set_information_matrix(choice_set,beta):
    p = N_ITEMS - 1
    info = np.zeros((p, p))

    # best choice information
    utilities = beta[list(choice_set)]
    best_probs = softmax(utilities)
    X = np.vstack([effect_vector(item, N_ITEMS) for item in choice_set])
    mean_x = (best_probs[:, None] * X).sum(axis=0)
    best_info = np.zeros((p, p))

    for j in range(len(choice_set)):
        d = X[j] - mean_x
        best_info += (best_probs[j] * np.outer(d, d))
    info += best_info
    
    # Expected worst choice information 
    for best_position in range(len(choice_set)):
        p_best = best_probs[best_position]
        remaining_positions = [i for i in range(len(choice_set)) if i != best_position]
        remaining_items = [choice_set[i] for i in remaining_positions]
        remaining_utilities = beta[remaining_items]

        # Worst:probability proportional to exp(-utility)
        worst_probs = softmax(-remaining_utilities)
        X_remaining = np.vstack([effect_vector(item,N_ITEMS) for item in remaining_items])

        # Because worst choice uses -utility, design vectors are effectively -X.
        X_worst = -X_remaining
        mean_x_worst = (worst_probs[:, None] * X_worst).sum(axis=0)
        worst_info = np.zeros((p, p))

        for j in range(len(remaining_items)):
            d = (X_worst[j] - mean_x_worst)
            worst_info += (worst_probs[j] * np.outer(d, d))

        info += (p_best * worst_info)

    return info

# D-optimality score
def d_score(info_matrix):
    # Tiny ridge only for numerical stability
    ridge = (np.eye(info_matrix.shape[0]) * 1e-9)
    sign, logdet = np.linalg.slogdet(info_matrix + ridge)
    if sign <= 0:
        return -np.inf
    return logdet

# Generate candidate Maxdiff sets
# For J=20 and K=4: C(20,4) = 4,845
candidate_sets = list(combinations(range(N_ITEMS), ITEMS_PER_TASK))
print("Candidate MaxDiff sets:",len(candidate_sets))

# precompute info for each candidate
candidate_information = {}
for choice_set in candidate_sets:
    candidate_information[choice_set] = set_information_matrix(choice_set, PRIOR_BETA)

# Create initial random design 
TOTAL_DESIGN_TASKS = (N_VERSIONS * TASKS_PER_VERSION)
initial_indices = rng.choice(len(candidate_sets), size = TOTAL_DESIGN_TASKS, replace = False)
current_design = [candidate_sets[i] for i in initial_indices]

# Information Matrix
def design_information(design):
    info = np.zeros((N_ITEMS - 1, N_ITEMS - 1))
    for choice_set in design:
        info += candidate_information[choice_set]
    return info

current_info = design_information(current_design)
current_score = d_score(current_info)
print("Initial D-score:", current_score)

# Exchange Optimizer
# Coordinate-exchange style search: Replace existing tasks with candidate tasks whenever the replacement improves log(det(I)).
MAX_PASSES = 10
CANDIDATES_PER_EXCHANGE = 300

for pass_number in range(MAX_PASSES):
    improved = False
    order = rng.permutation(TOTAL_DESIGN_TASKS)
    for position in order:
        old_set = current_design[position]
        info_without_old = (current_info - candidate_information[old_set])
        candidate_indices = rng.choice(len(candidate_sets), size = min(CANDIDATES_PER_EXCHANGE,len(candidate_sets)), replace = False)
        best_score = current_score
        best_set = old_set
        best_info = current_info

        for idx in candidate_indices:
            new_set = candidate_sets[idx]
            if new_set == old_set:
                continue
            proposed_info = (info_without_old + candidate_information[new_set])
            proposed_score = d_score(proposed_info)
            if proposed_score > best_score:
                best_score = (proposed_score)
                best_set = new_set
                best_info = proposed_info

        if best_set != old_set:
            current_design[position] = best_set
            current_info = best_info
            current_score = best_score
            improved = True

    print(f"Pass {pass_number + 1}: "
          f"D-score = {current_score:.6f}")

    if not improved:
        break

# Distrubute tasks into questionnaire versions
questionnaire_rows = []
for version in range(N_VERSIONS):
    start = (version * TASKS_PER_VERSION)
    end = (start + TASKS_PER_VERSION)
    version_sets = current_design[start:end]
    for task_number, choice_set in enumerate(version_sets, start = 1):
        # Randomize presentation position.
        displayed = list(choice_set)
        rng.shuffle(displayed)
        questionnaire_rows.append({
            "version": version + 1,
            "task": task_number,
            "option_1": ITEM_NAMES[displayed[0]],
            "option_2": ITEM_NAMES[displayed[1]],
            "option_3": ITEM_NAMES[displayed[2]],
            "option_4": ITEM_NAMES[displayed[3]],
            "canonical_set": "|".join(ITEM_NAMES[i] for i in sorted(choice_set))
        })
questionnaire = pd.DataFrame(questionnaire_rows)

# Respondents to questionnaire versions 
assignments = pd.DataFrame({"respondent_id": np.arange(1, N_RESPONDENTS + 1)})
assignments["version"] = ((assignments["respondent_id"] - 1) % N_VERSIONS) + 1

fieldwork = assignments.merge(questionnaire, on="version", how="left")
fieldwork = fieldwork.sort_values(["respondent_id", "task"])

# item balance
all_displayed_items = pd.concat([
    questionnaire["option_1"],
    questionnaire["option_2"],
    questionnaire["option_3"],
    questionnaire["option_4"]])
item_balance = (all_displayed_items
                .value_counts()
                .reindex(ITEM_NAMES, fill_value=0)
                .rename_axis("feature")
                .reset_index(name="exposures"))

# pair balance
pair_counts = {}
for choice_set in current_design:
    for pair in combinations(choice_set, 2):
        pair_counts[pair] = (pair_counts.get(pair, 0) + 1)
pair_balance = pd.DataFrame([{
        "feature_A":ITEM_NAMES[a],
        "feature_B":ITEM_NAMES[b],
        "co_occurrences":pair_counts.get((a, b), 0)
    } for a, b in combinations(range(N_ITEMS), 2)])

# connectivity check
adjacency = np.zeros((N_ITEMS, N_ITEMS), dtype=int)
for choice_set in current_design:
    for a, b in combinations(choice_set, 2):
        adjacency[a, b] = 1
        adjacency[b, a] = 1

def connected_graph(adjacency):
    visited = set()
    stack = [0]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        neighbors = np.where(adjacency[node] > 0)[0]
        stack.extend(n for n in neighbors if n not in visited)

    return (len(visited) == N_ITEMS)

print("\nConnected design:", connected_graph(adjacency))

questionnaire.to_csv("maxdiff_questionnaire.csv", index=False)
assignments.to_csv("respondent_version_assignments.csv", index=False)
fieldwork.to_csv("maxdiff_fieldwork.csv", index=False)
item_balance.to_csv("maxdiff_item_balance.csv", index=False)
pair_balance.to_csv("maxdiff_pair_balance.csv", index=False)

print("\nFinal D-score:", current_score)
print("\nQuestionnaire:")
print(questionnaire.head(12))
print("\nItem balance:")
print(item_balance)
print("\nPair balance:")
print(pair_balance.describe())

Candidate MaxDiff sets: 4845
Initial D-score: 57.5809242329404
Pass 1: D-score = 57.791462
Pass 2: D-score = 57.794215
Pass 3: D-score = 57.794215

Connected design: True

Final D-score: 57.7942146607339

Questionnaire:
    version  task    option_1    option_2    option_3    option_4  \
0         1     1  Feature_14  Feature_20  Feature_04  Feature_12   
1         1     2  Feature_11  Feature_13  Feature_06  Feature_17   
2         1     3  Feature_12  Feature_18  Feature_02  Feature_14   
3         1     4  Feature_06  Feature_20  Feature_08  Feature_19   
4         1     5  Feature_14  Feature_08  Feature_02  Feature_13   
5         1     6  Feature_09  Feature_13  Feature_02  Feature_16   
6         1     7  Feature_08  Feature_15  Feature_13  Feature_20   
7         1     8  Feature_14  Feature_16  Feature_13  Feature_11   
8         1     9  Feature_04  Feature_05  Feature_01  Feature_03   
9         1    10  Feature_14  Feature_20  Feature_08  Feature_13   
10        1    11  Fe

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from math import comb
from scipy.special import softmax

class MaxDiffDesigner:
    def __init__(
        self,
        n_items=20,
        items_per_task=4,
        tasks_per_version=12,
        n_versions=20,
        n_respondents=1000,
        item_names=None,
        prior_beta=None,
        seed=42,
        max_passes=5,
        candidates_per_exchange=200,
        max_candidates=50000
    ):
        self.J = n_items
        self.K = items_per_task
        self.T = tasks_per_version
        self.V = n_versions
        self.N = n_respondents

        self.rng = np.random.default_rng(seed)

        self.max_passes = max_passes
        self.candidates_per_exchange = candidates_per_exchange
        self.max_candidates = max_candidates

        if not 2 <= self.K <= self.J:
            raise ValueError("items_per_task must be between 2 and n_items.")

        self.names = (item_names if item_names is not None
                      else [f"Feature_{i+1:02d}" for i in range(self.J)])

        if len(self.names) != self.J:
            raise ValueError("item_names must have n_items entries.")
        
        if prior_beta is None:
            self.beta = np.zeros(self.J)
        else:
            self.beta = np.asarray(prior_beta, dtype=float)
            if len(self.beta) != self.J:
                raise ValueError("prior_beta must have n_items entries.")
            self.beta -= self.beta.mean()

    # Effect Coding
    def _x(self, item):
        x = np.zeros(self.J - 1)
        if item < self.J - 1:
            x[item] = 1
        else:
            x[:] = -1
        return x

    # Multinomial Fischer Information
    @staticmethod
    def _mnl_info(X, p):
        W = (np.diag(p) - np.outer(p, p))
        return X.T @ W @ X

    # Information from Maxdiff task
    def _task_info(self, choice_set):
        choice_set = tuple(choice_set)
        X = np.vstack([self._x(i) for i in choice_set])
        utilities = self.beta[list(choice_set)]

        # best
        p_best = softmax(utilities)
        info = self._mnl_info(X, p_best)

        # worst: expected info conditional on every possible best choice.
        for best_pos in range(self.K):
            remaining = [p for p in range(self.K) if p != best_pos]
            remaining_items = [choice_set[p] for p in remaining]
            X_remaining = np.vstack([self._x(i) for i in remaining_items])
            worst_utilities = (-self.beta[remaining_items])
            p_worst = softmax(worst_utilities)
            worst_info = self._mnl_info(-X_remaining, p_worst)
            info += (p_best[best_pos] * worst_info)
        return info

    # D-optimality score
    def _score(self, info):
        sign, logdet = np.linalg.slogdet(info + np.eye(self.J - 1) * 1e-9)
        return (logdet if sign > 0 else -np.inf)

    # Generate candidates
    def _make_candidates(self):
        total = comb(self.J, self.K)
        if total <= self.max_candidates:
            candidates = list(combinations(range(self.J), self.K))
        else:
            candidate_set = set()
            while (len(candidate_set) < self.max_candidates):
                candidate = tuple(sorted(self.rng.choice(self.J, self.K, replace=False)))
                candidate_set.add(candidate)
            candidates = list(candidate_set)
        return candidates

    # Fit the design
    def fit(self):
        print(f"Generating design: "
              f"{self.J} items, "
              f"{self.K}/task, "
              f"{self.T} tasks/version, "
              f"{self.V} versions"
              )

        # Candidate sets
        self.candidates = (self._make_candidates())
        print(f"Candidate sets: "
              f"{len(self.candidates):,}"
              )

        # Precalculate information
        self.info = {c: self._task_info(c) for c in self.candidates}

        # Initial design
        n_design_tasks = (self.T * self.V)
        replace = (n_design_tasks > len(self.candidates))
        indices = self.rng.choice(len(self.candidates), n_design_tasks, replace=replace)
        design = [self.candidates[i] for i in indices]
        total_info = sum((self.info[c]for c in design), 
                         start=np.zeros((self.J - 1, self.J - 1)))
        initial_score = self._score(total_info)
        score = initial_score
        print(f"Initial D-score: "
              f"{score:.4f}")

        # Exchange optimization
        for iteration in range(self.max_passes):
            changes = 0
            for pos in self.rng.permutation(n_design_tasks):
                old = design[pos]
                without_old = (total_info - self.info[old])
                n_try = min(self.candidates_per_exchange, len(self.candidates))
                test_indices = (self.rng.choice(len(self.candidates), n_try, replace=False))
                best_candidate = old
                best_info = total_info
                best_score = score

                for idx in test_indices:
                    candidate = (self.candidates[idx])
                    proposed_info = (without_old + self.info[candidate])
                    proposed_score = (self._score(proposed_info))
                    if (proposed_score > best_score + 1e-10):
                        best_candidate = (candidate)
                        best_info = (proposed_info)
                        best_score = (proposed_score)
                
                if best_candidate != old:
                    design[pos] = (best_candidate)
                    total_info = (best_info)
                    score = (best_score)
                    changes += 1

            print(f"Pass {iteration + 1}: "
                  f"D-score={score:.4f}, "
                  f"changes={changes}")

            if changes == 0:
                break

        self.design = design
        self.information_matrix = total_info
        self.initial_d_score = (initial_score)
        self.final_d_score = (score)

        # Build questionnaire
        rows = []
        for version in range(self.V):
            sets = design[version * self.T:(version + 1) * self.T]
            for task, choice_set in enumerate(sets, start=1):
                display = list(choice_set)
                self.rng.shuffle(display)
                row = {
                    "version": version + 1, 
                    "task": task
                    }
                for p, item in enumerate(display, start=1):
                    row[f"option_{p}"] = (self.names[item])
                rows.append(row)

        self.questionnaire = (pd.DataFrame(rows))

        # Respondent assignement
        assignments = pd.DataFrame({"respondent_id":np.arange(1, self.N + 1)})
        assignments["version"] = ((assignments["respondent_id"] - 1) % self.V) + 1
        self.assignments = assignments
        self.fieldwork = (assignments.merge(self.questionnaire, on="version"))

        # item balance
        item_counts = np.zeros(self.J, dtype=int)
        for choice_set in design:
            for item in choice_set:
                item_counts[item] += 1
        item_target = (len(design) * self.K / self.J)
        self.item_balance = pd.DataFrame({
            "item": self.names,
            "exposure": item_counts,
            "target": item_target,
            "deviation": item_counts - item_target
            })

        # pair balance
        pair_counts = {}
        for choice_set in design:
            for pair in combinations(choice_set, 2):
                pair_counts[pair] = (pair_counts.get(pair, 0) + 1)
        
        total_pair_exposures = (len(design) * comb(self.K, 2))
        pair_target = (total_pair_exposures / comb(self.J, 2))
        pair_rows = []
        
        for a, b in combinations(range(self.J), 2):
            count = pair_counts.get((a, b), 0)
            pair_rows.append({
                "item_A": self.names[a],
                "item_B": self.names[b],
                "exposure": count,
                "target": pair_target,
                "deviation": count - pair_target
                })

        self.pair_balance = (pd.DataFrame(pair_rows))

        # connectivity
        adjacency = {i: set() for i in range(self.J)}
        for choice_set in design:
            for a, b in combinations(choice_set, 2):
                adjacency[a].add(b)
                adjacency[b].add(a)

        visited = set()
        stack = [0]
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            stack.extend(adjacency[node] - visited)
        self.connected = (len(visited) == self.J)

        self.summary = pd.DataFrame([{
            "items": self.J,
            "items_per_task": self.K,
            "tasks_per_version": self.T,
            "versions": self.V,
            "respondents": self.N,
            "initial_d_score": self.initial_d_score,
            "final_d_score": self.final_d_score,
            "d_score_gain": self.final_d_score - self.initial_d_score,
            "min_item_exposure": item_counts.min(),
            "max_item_exposure": item_counts.max(),
            "target_item_exposure": item_target,
            "min_pair_exposure": self.pair_balance["exposure"].min(),
            "max_pair_exposure": self.pair_balance["exposure"].max(),
            "target_pair_exposure": pair_target,
            "connected": self.connected
            }])

        print("\nDesign complete.")
        print(f"Final D-score: "
              f"{score:.4f}")
        print(f"Connected: "
              f"{self.connected}")
        
        return self

    def show_version(self, version=1):
        return (self.questionnaire[self.questionnaire["version"] == version].reset_index(drop=True))

    def export(self, prefix="maxdiff"):
        self.questionnaire.to_csv(f"{prefix}_questionnaire.csv", index=False)
        self.assignments.to_csv(f"{prefix}_assignments.csv", index=False)
        self.fieldwork.to_csv(f"{prefix}_fieldwork.csv", index=False)
        self.item_balance.to_csv(f"{prefix}_item_balance.csv", index=False)
        self.pair_balance.to_csv(f"{prefix}_pair_balance.csv", index=False)
        self.summary.to_csv(f"{prefix}_summary.csv", index=False)

# Example:
# 20 items
# 4 items/question
# 12 questions/respondent
# 20 questionnaire versions
# 1,000 respondents
experiment = MaxDiffDesigner(
    n_items=20,
    items_per_task=4,
    tasks_per_version=12,
    n_versions=20,
    n_respondents=1000,
    seed=42
)
experiment.fit()

print("\nQUESTIONNAIRE VERSION 1")
print(experiment.show_version(1))
print("\nDESIGN SUMMARY")
print(experiment.summary.T)
print("\nITEM BALANCE")
print(experiment.item_balance)
print("\nPAIR BALANCE")
print(experiment.pair_balance)
# experiment.export("experiment_1")

Generating design: 20 items, 4/task, 12 tasks/version, 20 versions
Candidate sets: 4,845
Initial D-score: 57.5809
Pass 1: D-score=57.7933, changes=114
Pass 2: D-score=57.7948, changes=12
Pass 3: D-score=57.7948, changes=0

Design complete.
Final D-score: 57.7948
Connected: True

QUESTIONNAIRE VERSION 1
    version  task    option_1    option_2    option_3    option_4
0         1     1  Feature_01  Feature_06  Feature_18  Feature_05
1         1     2  Feature_17  Feature_13  Feature_11  Feature_06
2         1     3  Feature_14  Feature_16  Feature_18  Feature_10
3         1     4  Feature_08  Feature_06  Feature_19  Feature_20
4         1     5  Feature_03  Feature_13  Feature_14  Feature_06
5         1     6  Feature_06  Feature_13  Feature_09  Feature_14
6         1     7  Feature_08  Feature_13  Feature_09  Feature_15
7         1     8  Feature_13  Feature_16  Feature_14  Feature_11
8         1     9  Feature_03  Feature_10  Feature_05  Feature_01
9         1    10  Feature_16  Featu

In [4]:
experiment_2 = MaxDiffDesigner(
    n_items=15,
    items_per_task=3,
    tasks_per_version=10,
    n_versions=15,
    n_respondents=500,
    seed=42
)

experiment_2.fit()

print(experiment_2.summary.T)

Generating design: 15 items, 3/task, 10 tasks/version, 15 versions
Candidate sets: 455
Initial D-score: 37.8924
Pass 1: D-score=38.0573, changes=65
Pass 2: D-score=38.0573, changes=0

Design complete.
Final D-score: 38.0573
Connected: True
                              0
items                        15
items_per_task                3
tasks_per_version            10
versions                     15
respondents                 500
initial_d_score       37.892447
final_d_score         38.057299
d_score_gain           0.164852
min_item_exposure            30
max_item_exposure            30
target_item_exposure       30.0
min_pair_exposure             3
max_pair_exposure             6
target_pair_exposure   4.285714
connected                  True


In [5]:
experiment_3 = MaxDiffDesigner(
    n_items=30,
    items_per_task=5,
    tasks_per_version=15,
    n_versions=30,
    n_respondents=2000,
    seed=42
)

experiment_3.fit()

Generating design: 30 items, 5/task, 15 tasks/version, 30 versions
Candidate sets: 50,000
Initial D-score: 95.3390
Pass 1: D-score=95.6163, changes=208
Pass 2: D-score=95.6200, changes=43
Pass 3: D-score=95.6205, changes=9
Pass 4: D-score=95.6208, changes=5
Pass 5: D-score=95.6209, changes=2

Design complete.
Final D-score: 95.6209
Connected: True
